# Notebook 2 — Forecast Strategy
#### Build the baseline forecast, apply seasonal event uplifts, and generate the 2026 cocktail demand forecast.

In [1]:
from pathlib import Path

# ✅ point this at the SAME db file you built in Notebook 1
DB_PATH = Path("../db/cocktail_engine.db")   # <- change if your db lives elsewhere

print("✅ Using DB:", DB_PATH.resolve())

✅ Using DB: /Users/carolineridgway/workbench/cocktail-engine/db/cocktail_engine.db


In [2]:
import sqlite3, pandas as pd

with sqlite3.connect(DB_PATH) as con:
    pd.read_sql("""
        SELECT year, month, SUM(reality_forecast_qty) AS total_qty
        FROM fact_reality_forecast_14f
        GROUP BY year, month
        ORDER BY year, month
        LIMIT 12;
    """, con)

## 7) 2026 Baseline Forecast Engine



#### 7A) Extract 2025 Seasonality Weights

In [3]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_2025_seasonality_weights_7a;")
    cur.execute("""
        CREATE VIEW v_2025_seasonality_weights_7a AS
        WITH totals AS (
            SELECT
                cocktail_id,
                SUM(reality_forecast_qty) AS total_2025_qty
            FROM fact_reality_forecast_14f
            WHERE year = 2025
            GROUP BY cocktail_id
        )
        SELECT
            r.cocktail_id,
            r.cocktail_name,
            r.month,
            r.reality_forecast_qty,
            t.total_2025_qty,
            ROUND(
                r.reality_forecast_qty * 1.0 / NULLIF(t.total_2025_qty, 0),
                6
            ) AS seasonal_weight
        FROM fact_reality_forecast_14f r
        JOIN totals t
            ON t.cocktail_id = r.cocktail_id
        WHERE r.year = 2025;
    """)

print("✅ 7A — seasonality weights created")

✅ 7A — seasonality weights created


#### 7B) Project 2026 Annual Totals

In [4]:
import sqlite3

BASE_GROWTH = 1.05

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_2026_projected_totals_7b;")
    cur.execute(f"""
        CREATE VIEW v_2026_projected_totals_7b AS
        SELECT
            cocktail_id,
            MAX(cocktail_name) AS cocktail_name,
            SUM(reality_forecast_qty) * {BASE_GROWTH} AS projected_2026_total_qty
        FROM fact_reality_forecast_14f
        WHERE year = 2025
        GROUP BY cocktail_id;
    """)

print("✅ 7B — projected totals created")

✅ 7B — projected totals created


#### 7C) Redistribute 2026 Baseline

In [5]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_2026_baseline_qty_7c;")
    cur.execute("""
        CREATE VIEW v_2026_baseline_qty_7c AS
        WITH months AS (
            SELECT 2026 AS year, month
            FROM dim_month_14a
            WHERE year = 2026 AND month BETWEEN 1 AND 12
        ),
        cocktails AS (
            SELECT cocktail_id FROM dim_cocktail
        ),
        -- seasonal pattern from 2025 if it exists in fact_reality_forecast_14f
        seasonal_2025 AS (
            SELECT
                cocktail_id,
                month,
                AVG(reality_forecast_qty) AS seasonal_qty
            FROM fact_reality_forecast_14f
            WHERE year = 2025
            GROUP BY cocktail_id, month
        ),
        -- fallback: overall average per cocktail from whatever exists in 14f
        fallback_mean AS (
            SELECT
                cocktail_id,
                AVG(reality_forecast_qty) AS mean_qty
            FROM fact_reality_forecast_14f
            GROUP BY cocktail_id
        )
        SELECT
            c.cocktail_id,
            m.year,
            m.month,
            COALESCE(s.seasonal_qty, f.mean_qty, 0) AS baseline_2026_qty
        FROM cocktails c
        CROSS JOIN months m
        LEFT JOIN seasonal_2025 s
            ON s.cocktail_id = c.cocktail_id
           AND s.month = m.month
        LEFT JOIN fallback_mean f
            ON f.cocktail_id = c.cocktail_id;
    """)

    con.commit()

print("✅ 7C rebuilt from fact_reality_forecast_14f (always 12 months)")

✅ 7C rebuilt from fact_reality_forecast_14f (always 12 months)


#### 7D) Persist 2026 Baseline 

In [6]:
import sqlite3
import pandas as pd
import numpy as np

# ---- CONFIG ----
SCAFFOLD_START = "2024-12-01"
SCAFFOLD_END   = "2026-11-01"

with sqlite3.connect(DB_PATH) as con:
    # Pull monthly actuals (adjust column names here if your fact table differs)
    sales = pd.read_sql("""
        SELECT
            c.cocktail_id,
            c.cocktail_name,
            s.year,
            s.month,
            s.quantity_sold AS qty
        FROM fact_monthly_sales s
        JOIN dim_cocktail c
          ON c.cocktail_id = s.cocktail_id
        ORDER BY c.cocktail_name, s.year, s.month
    """, con)

# Build date
sales["date"] = pd.to_datetime(sales["year"].astype(str) + "-" + sales["month"].astype(str) + "-01")

# Full scaffold dates
scaffold_dates = pd.date_range(SCAFFOLD_START, SCAFFOLD_END, freq="MS")

# Expand each cocktail across the full scaffold
cocktails = sales[["cocktail_id", "cocktail_name"]].drop_duplicates()

scaffold = (
    cocktails.assign(key=1)
    .merge(pd.DataFrame({"date": scaffold_dates, "key": 1}), on="key")
    .drop(columns=["key"])
)

# Merge actuals onto scaffold
df = scaffold.merge(
    sales[["cocktail_id", "date", "qty"]],
    on=["cocktail_id", "date"],
    how="left"
)

# If a month is missing in actuals, treat as 0 for rolling calc
df["qty"] = df["qty"].fillna(0)

# MA3 baseline per cocktail (rolling mean)
df = df.sort_values(["cocktail_id", "date"])
df["baseline_ma3"] = (
    df.groupby("cocktail_id")["qty"]
      .rolling(3, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

# Forward-fill baseline into future months (so 2026 has values even if qty=0)
df["baseline_ma3"] = df.groupby("cocktail_id")["baseline_ma3"].ffill()

# Split year/month for storage
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

# Final table
df_baseline = df[["cocktail_id", "cocktail_name", "year", "month", "baseline_ma3"]].copy()

with sqlite3.connect(DB_PATH) as con:
    # Create/replace the baseline table
    con.execute("DROP TABLE IF EXISTS fact_2026_baseline_7d;")
    con.execute("""
        CREATE TABLE fact_2026_baseline_7d (
            cocktail_id   INTEGER,
            cocktail_name TEXT,
            year          INTEGER,
            month         INTEGER,
            baseline_ma3  REAL
        );
    """)
    df_baseline.to_sql("fact_2026_baseline_7d", con, if_exists="append", index=False)

print("✅ Recreated fact_2026_baseline_7d:", len(df_baseline), "rows")
df_baseline.tail(12)

✅ Recreated fact_2026_baseline_7d: 672 rows


,cocktail_id,cocktail_name,year,month,baseline_ma3
516,133,Ship's Rumbullion,2025,12,0.0
517,133,Ship's Rumbullion,2026,1,0.0
518,133,Ship's Rumbullion,2026,2,0.0
519,133,Ship's Rumbullion,2026,3,0.0
520,133,Ship's Rumbullion,2026,4,0.0
521,133,Ship's Rumbullion,2026,5,0.0
522,133,Ship's Rumbullion,2026,6,0.0
523,133,Ship's Rumbullion,2026,7,0.0
524,133,Ship's Rumbullion,2026,8,0.0
525,133,Ship's Rumbullion,2026,9,0.0


In [7]:
import pandas as pd
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    df = pd.read_sql("""
        SELECT MIN(month) AS min_m, MAX(month) AS max_m, COUNT(DISTINCT month) AS n_months
        FROM v_2026_baseline_qty_7c
        WHERE year=2026;
    """, con)

df

,min_m,max_m,n_months
0,1,11,11


In [8]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("""
        INSERT OR IGNORE INTO dim_month_14a (year, month)
        VALUES (2026, 12);
    """)

    con.commit()

print("✅ December 2026 added to scaffold")

✅ December 2026 added to scaffold


#### 7E — Quick sanity checks 

In [9]:
import pandas as pd
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    df = pd.read_sql("""
        SELECT COUNT(*) AS rows_2026
        FROM fact_2026_baseline_7d
        WHERE year = 2026;
    """, con)

df

,rows_2026
0,308


## 8: Recurring Event Model

#### 8A) Crate Annualised Driver Template

In [10]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP TABLE IF EXISTS fact_annual_driver_template_8a;")
    cur.execute("""
        CREATE TABLE fact_annual_driver_template_8a (
            template_month INTEGER NOT NULL,     -- 1..12
            driver_type TEXT NOT NULL,           -- ROYAL / GARDEN / COCKTAIL / SEASON
            target_scope TEXT NOT NULL DEFAULT 'ALL',
            target_value TEXT,                  -- NULL allowed
            target_value_key TEXT NOT NULL DEFAULT '',  -- store '' when target_value is NULL
            uplift REAL NOT NULL,               -- e.g. 0.20 = +20%
            notes TEXT,
            PRIMARY KEY (template_month, driver_type, target_scope, target_value_key)
        );
    """)

    con.commit()

print("✅ 8A — fact_annual_driver_template_8a created (SQLite-safe PK)")

✅ 8A — fact_annual_driver_template_8a created (SQLite-safe PK)


#### 8A-1 — Seed Template

In [11]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DELETE FROM fact_annual_driver_template_8a;")

    rows = [
        # month, type, scope, value, uplift, notes
        (1,  "SEASON",  "ALL", None, -0.25, "Dry January dip"),

        (5,  "GARDEN",  "ALL", None,  0.15, "Garden season uplift"),
        (6,  "GARDEN",  "ALL", None,  0.15, "Garden season uplift"),
        (7,  "GARDEN",  "ALL", None,  0.15, "Garden season uplift"),

        (6,  "COCKTAIL","ALL", None,  0.10, "Negroni Month uplift (broad bar effect)"),

        (7,  "SEASON",  "ALL", None,  0.10, "Summer peak uplift"),
        (8,  "SEASON",  "ALL", None,  0.10, "Summer peak uplift"),

        (11, "SEASON",  "ALL", None,  0.15, "Christmas uplift"),
        (12, "SEASON",  "ALL", None,  0.20, "Christmas uplift (stronger December)")
    ]

    cur.executemany("""
        INSERT INTO fact_annual_driver_template_8a
        (template_month, driver_type, target_scope, target_value, target_value_key, uplift, notes)
        VALUES (?, ?, ?, ?, COALESCE(?, ''), ?, ?);
    """, [(m,t,s,v,v,u,n) for (m,t,s,v,u,n) in rows])

    con.commit()

print("✅ 8A-1 — Annual driver template seeded")

✅ 8A-1 — Annual driver template seeded


In [12]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS fact_demand_drivers_2026_8b (
            driver_date TEXT NOT NULL,     -- YYYY-MM-01
            driver_type TEXT NOT NULL,
            target_scope TEXT NOT NULL,
            target_value TEXT,
            uplift REAL NOT NULL,
            notes TEXT
        );
    """)

    cur.execute("DELETE FROM fact_demand_drivers_2026_8b;")

    cur.execute("""
        INSERT INTO fact_demand_drivers_2026_8b
        (driver_date, driver_type, target_scope, target_value, uplift, notes)
        SELECT
            printf('2026-%02d-01', template_month) AS driver_date,
            driver_type,
            target_scope,
            target_value,
            uplift,
            notes
        FROM fact_annual_driver_template_8a;
    """)

    con.commit()

print("✅ 8B — 2026 demand drivers generated")

✅ 8B — 2026 demand drivers generated


#### 8C) Aggregate 2026 Monthly Uplift

In [13]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS fact_month_driver_uplift_2026_8c (
            year INTEGER NOT NULL,
            month INTEGER NOT NULL,
            total_uplift REAL NOT NULL,
            PRIMARY KEY (year, month)
        );
    """)

    cur.execute("DELETE FROM fact_month_driver_uplift_2026_8c WHERE year = 2026;")

    cur.execute("""
        INSERT INTO fact_month_driver_uplift_2026_8c (year, month, total_uplift)
        SELECT
            2026 AS year,
            CAST(strftime('%m', driver_date) AS INT) AS month,
            ROUND(SUM(uplift), 4) AS total_uplift
        FROM fact_demand_drivers_2026_8b
        GROUP BY CAST(strftime('%m', driver_date) AS INT);
    """)

    con.commit()

print("✅ 8C — 2026 monthly uplift aggregated (non-compounding SUM)")

✅ 8C — 2026 monthly uplift aggregated (non-compounding SUM)


#### 8D) Apply 2026 Uplift to 2026 Baseline

In [14]:
import sqlite3

UPLIFT_COL = "total_uplift"   # this is correct for fact_month_driver_uplift_2026_8c in your notebook
YOY_GROWTH = 0.15
GROWTH_FACTOR_2026 = 1 + YOY_GROWTH   # 1.15

with sqlite3.connect(DB_PATH) as con:
    con.execute("DROP VIEW IF EXISTS v_2026_event_adjusted_qty_8d;")
    con.execute(f"""
    CREATE VIEW v_2026_event_adjusted_qty_8d AS
    SELECT
        b.cocktail_id,
        b.year,
        b.month,
        b.baseline_ma3 AS baseline_2026_qty,
        COALESCE(d.{UPLIFT_COL}, 0) AS uplift,
        {GROWTH_FACTOR_2026} AS growth_factor,
        ROUND(
            b.baseline_ma3
            * (1 + COALESCE(d.{UPLIFT_COL}, 0))
            * {GROWTH_FACTOR_2026}
        , 2) AS event_adjusted_2026_qty
    FROM fact_2026_baseline_7d b
    LEFT JOIN fact_month_driver_uplift_2026_8c d
      ON d.year = b.year
     AND d.month = b.month;
    """)

print("✅ 8D rebuilt: baseline × (1 + uplift) × growth_factor (1.15)")

✅ 8D rebuilt: baseline × (1 + uplift) × growth_factor (1.15)


#### 8E) Persist 2026 Event Forecast

In [15]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:

    con.execute("""
        CREATE TABLE IF NOT EXISTS fact_2026_event_forecast_8e (
            cocktail_id INTEGER,
            year INTEGER,
            month INTEGER,
            baseline_2026_qty REAL,
            uplift REAL,
            event_adjusted_2026_qty REAL
        );
    """)

    con.execute("DELETE FROM fact_2026_event_forecast_8e WHERE year = 2026;")

    con.execute("""
        INSERT INTO fact_2026_event_forecast_8e
        SELECT
            cocktail_id,
            year,
            month,
            baseline_2026_qty,
            uplift,
            event_adjusted_2026_qty
        FROM v_2026_event_adjusted_qty_8d;
    """)

print("✅ 8E persisted (now includes growth)")

✅ 8E persisted (now includes growth)


## Archived — Financial Layer (8F/8G)
### These were early financial calculations applied directly to the event-adjusted forecast.
### The final portfolio uses the unified reality financial layer (14F/14G) instead.

#### 8F) 2026 Financial Layer

In [16]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_2026_financial_8f;")
    cur.execute("""
        CREATE VIEW v_2026_financial_8f AS
        SELECT
            f.cocktail_id,
            c.cocktail_name,
            f.year,
            f.month,
            f.baseline_2026_qty,
            f.uplift,
            f.event_adjusted_2026_qty,

            p.sell_price,
            cp.cost_per_cocktail,

            ROUND(f.event_adjusted_2026_qty * p.sell_price, 2) AS forecast_revenue,
            ROUND(f.event_adjusted_2026_qty * cp.cost_per_cocktail, 2) AS forecast_cogs,

            ROUND(
                (f.event_adjusted_2026_qty * p.sell_price)
                -
                (f.event_adjusted_2026_qty * cp.cost_per_cocktail),
            2) AS forecast_gross_profit

        FROM fact_2026_event_forecast_8e f
        LEFT JOIN dim_cocktail c
            ON c.cocktail_id = f.cocktail_id
        LEFT JOIN fact_cocktail_price p
            ON p.cocktail_id = f.cocktail_id
        LEFT JOIN v_cost_per_cocktail cp
            ON cp.cocktail_id = f.cocktail_id;
    """)

print("✅ 8F — v_2026_financial_8f created")

✅ 8F — v_2026_financial_8f created


#### 8G — 2026 Monthly Executive Summary

In [17]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_2026_exec_monthly_8g;")
    cur.execute("""
        CREATE VIEW v_2026_exec_monthly_8g AS
        SELECT
            year,
            month,
            SUM(event_adjusted_2026_qty) AS total_qty,
            ROUND(SUM(forecast_revenue), 2) AS total_revenue,
            ROUND(SUM(forecast_cogs), 2) AS total_cogs,
            ROUND(SUM(forecast_gross_profit), 2) AS total_gp
        FROM v_2026_financial_8f
        GROUP BY year, month
        ORDER BY year, month;
    """)

print("✅ 8G — v_2026_exec_monthly_8g created")

✅ 8G — v_2026_exec_monthly_8g created


In [18]:
import pandas as pd
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    df = pd.read_sql("SELECT * FROM v_2026_exec_monthly_8g;", con)

df

,year,month,total_qty,total_revenue,total_cogs,total_gp
0,2024,12,2063.10,41802.50,4238.91,36172.28
1,2025,1,1543.82,31245.57,3139.68,26890.24
2,2025,2,1359.30,27749.18,2789.89,23813.87
3,2025,3,1088.66,22449.51,2180.43,18876.15
4,2025,4,1112.06,23113.04,2238.92,19361.50
5,2025,5,782.00,16194.56,1542.16,13474.64
6,2025,6,364.93,7560.24,738.63,6355.17
7,2025,7,0.00,0.00,0.00,0.00
8,2025,8,0.00,0.00,0.00,0.00
9,2025,9,0.00,0.00,0.00,0.00


### 9) Price & Cost Inflation Layer

    Price uplift: +3%, +5%

    Cost inflation: +4%, +7%
    
    Output: revenue / cogs / gp under each combo
    
    Plus: margin compression view (delta GP vs base)

#### 9A ) Scenario Rate Table

In [19]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS dim_inflation_scenario_9a (
            scenario_name TEXT PRIMARY KEY,
            price_multiplier REAL NOT NULL,
            cost_multiplier REAL NOT NULL,
            notes TEXT
        );
    """)

    # reset seed
    cur.execute("DELETE FROM dim_inflation_scenario_9a;")

    cur.execute("""
        INSERT INTO dim_inflation_scenario_9a (scenario_name, price_multiplier, cost_multiplier, notes)
        VALUES
            ('BASE_2026',           1.00, 1.00, 'No inflation applied'),
            ('PRICE_UP_3',          1.03, 1.00, '+3% price only'),
            ('PRICE_UP_5',          1.05, 1.00, '+5% price only'),
            ('COST_UP_4',           1.00, 1.04, '+4% cost only'),
            ('COST_UP_7',           1.00, 1.07, '+7% cost only'),
            ('PRICE3_COST4',        1.03, 1.04, '+3% price, +4% cost'),
            ('PRICE3_COST7',        1.03, 1.07, '+3% price, +7% cost'),
            ('PRICE5_COST4',        1.05, 1.04, '+5% price, +4% cost'),
            ('PRICE5_COST7',        1.05, 1.07, '+5% price, +7% cost');
    """)

    con.commit()

print("✅ 9A — dim_inflation_scenario_9a created + seeded")

✅ 9A — dim_inflation_scenario_9a created + seeded


#### 9B) Apply Inflation to 2026 Financials

In [20]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_2026_inflation_financial_9b;")
    cur.execute("""
        CREATE VIEW v_2026_inflation_financial_9b AS
        SELECT
            s.scenario_name,

            f.cocktail_id,
            c.cocktail_name,
            f.year,
            f.month,

            f.event_adjusted_2026_qty AS forecast_qty,

            p.sell_price AS base_sell_price,
            cp.cost_per_cocktail AS base_cost_per_cocktail,

            ROUND(p.sell_price * s.price_multiplier, 2) AS scenario_sell_price,
            ROUND(cp.cost_per_cocktail * s.cost_multiplier, 4) AS scenario_cost_per_cocktail,

            ROUND(f.event_adjusted_2026_qty * (p.sell_price * s.price_multiplier), 2) AS forecast_revenue,
            ROUND(f.event_adjusted_2026_qty * (cp.cost_per_cocktail * s.cost_multiplier), 2) AS forecast_cogs,

            ROUND(
                (f.event_adjusted_2026_qty * (p.sell_price * s.price_multiplier)) -
                (f.event_adjusted_2026_qty * (cp.cost_per_cocktail * s.cost_multiplier)),
            2) AS forecast_gross_profit

        FROM fact_2026_event_forecast_8e f
        CROSS JOIN dim_inflation_scenario_9a s
        LEFT JOIN dim_cocktail c
            ON c.cocktail_id = f.cocktail_id
        LEFT JOIN fact_cocktail_price p
            ON p.cocktail_id = f.cocktail_id
        LEFT JOIN v_cost_per_cocktail cp
            ON cp.cocktail_id = f.cocktail_id;
    """)

print("✅ 9B — v_2026_inflation_financial_9b created")

✅ 9B — v_2026_inflation_financial_9b created


#### 9C) Executive Summary by Scenario (monthly + annual)

##### 9C-1) Monthly totals per scenario

In [21]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_2026_inflation_exec_monthly_9c;")
    cur.execute("""
        CREATE VIEW v_2026_inflation_exec_monthly_9c AS
        SELECT
            scenario_name,
            year,
            month,
            SUM(forecast_qty) AS total_qty,
            ROUND(SUM(forecast_revenue), 2) AS total_revenue,
            ROUND(SUM(forecast_cogs), 2) AS total_cogs,
            ROUND(SUM(forecast_gross_profit), 2) AS total_gp
        FROM v_2026_inflation_financial_9b
        GROUP BY scenario_name, year, month
        ORDER BY scenario_name, year, month;
    """)

print("✅ 9C-1 — monthly scenario executive view created")

✅ 9C-1 — monthly scenario executive view created


##### 9C-2) Annual totals per scenario

In [22]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_2026_inflation_exec_annual_9c;")
    cur.execute("""
        CREATE VIEW v_2026_inflation_exec_annual_9c AS
        SELECT
            scenario_name,
            SUM(forecast_qty) AS total_qty,
            ROUND(SUM(forecast_revenue), 2) AS total_revenue,
            ROUND(SUM(forecast_cogs), 2) AS total_cogs,
            ROUND(SUM(forecast_gross_profit), 2) AS total_gp
        FROM v_2026_inflation_financial_9b
        GROUP BY scenario_name
        ORDER BY total_gp DESC;
    """)

print("✅ 9C-2 — annual scenario executive view created")

✅ 9C-2 — annual scenario executive view created


#### 9D) Margin Compression

In [23]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_2026_margin_compression_9d;")
    cur.execute("""
        CREATE VIEW v_2026_margin_compression_9d AS
        WITH base AS (
            SELECT
                cocktail_id,
                year,
                month,
                forecast_gross_profit AS base_gp
            FROM v_2026_inflation_financial_9b
            WHERE scenario_name = 'BASE_2026'
        )
        SELECT
            f.scenario_name,
            f.cocktail_id,
            f.cocktail_name,
            f.year,
            f.month,
            f.forecast_gross_profit,
            b.base_gp,
            ROUND(f.forecast_gross_profit - b.base_gp, 2) AS delta_gp
        FROM v_2026_inflation_financial_9b f
        JOIN base b
            ON b.cocktail_id = f.cocktail_id
           AND b.year = f.year
           AND b.month = f.month
        WHERE f.scenario_name <> 'BASE_2026';
    """)

print("✅ 9D — margin compression view created (delta vs BASE_2026)")

✅ 9D — margin compression view created (delta vs BASE_2026)


#### 9E) Quick outputs

import pandas as pd
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    display(pd.read_sql("SELECT * FROM v_2026_inflation_exec_annual_9c;", con).head(20))

import pandas as pd
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    display(pd.read_sql("""
        SELECT scenario_name, SUM(delta_gp) AS annual_delta_gp
        FROM v_2026_margin_compression_9d
        GROUP BY scenario_name
        ORDER BY annual_delta_gp;
    """, con))

#### 10) Scenario Engne

##### 10A) Scenario Engine

In [24]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS dim_scenario_10a (
            scenario_name TEXT PRIMARY KEY,
            demand_multiplier REAL NOT NULL,
            price_multiplier REAL NOT NULL,
            cost_multiplier REAL NOT NULL,
            notes TEXT
        );
    """)

    # reset seed
    cur.execute("DELETE FROM dim_scenario_10a;")

    cur.execute("""
        INSERT INTO dim_scenario_10a (scenario_name, demand_multiplier, price_multiplier, cost_multiplier, notes)
        VALUES
            ('BASE',                 1.00, 1.00, 1.00, 'Baseline: event-adjusted demand, current prices/costs'),
            ('CONSERVATIVE',         0.95, 1.00, 1.02, 'Softer demand (-5%), slight cost pressure'),
            ('AGGRESSIVE',           1.10, 1.03, 1.02, 'High occupancy (+10%), +3% price, mild cost rise'),
            ('HIGH_OCCUPANCY',       1.15, 1.00, 1.02, 'Strong footfall (+15%), stable price, mild cost rise'),
            ('INFLATION_PRESSURE',   1.00, 1.03, 1.07, 'Price +3% but costs +7% (margin compression)'),
            ('PRICE_ACTION',         1.00, 1.05, 1.04, 'Management pushes +5% price; costs +4%'),
            ('DOWNTURN',             0.90, 1.00, 1.04, 'Demand shock (-10%) with cost inflation');
    """)

    con.commit()

print("✅ 10A — dim_scenario_10a created + seeded")

✅ 10A — dim_scenario_10a created + seeded


#### 10B) Scenario Forecast

In [25]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_2026_scenario_forecast_10b;")
    cur.execute("""
        CREATE VIEW v_2026_scenario_forecast_10b AS
        SELECT
            s.scenario_name,

            f.cocktail_id,
            c.cocktail_name,
            f.year,
            f.month,

            -- demand
            f.event_adjusted_2026_qty AS base_qty,
            ROUND(f.event_adjusted_2026_qty * s.demand_multiplier) AS scenario_qty,

            -- base prices/costs
            p.sell_price AS base_sell_price,
            cp.cost_per_cocktail AS base_cost_per_cocktail,

            -- scenario prices/costs
            ROUND(p.sell_price * s.price_multiplier, 2) AS scenario_sell_price,
            ROUND(cp.cost_per_cocktail * s.cost_multiplier, 4) AS scenario_cost_per_cocktail,

            -- scenario financials
            ROUND((f.event_adjusted_2026_qty * s.demand_multiplier) * (p.sell_price * s.price_multiplier), 2) AS scenario_revenue,
            ROUND((f.event_adjusted_2026_qty * s.demand_multiplier) * (cp.cost_per_cocktail * s.cost_multiplier), 2) AS scenario_cogs,

            ROUND(
                ((f.event_adjusted_2026_qty * s.demand_multiplier) * (p.sell_price * s.price_multiplier)) -
                ((f.event_adjusted_2026_qty * s.demand_multiplier) * (cp.cost_per_cocktail * s.cost_multiplier)),
            2) AS scenario_gross_profit

        FROM fact_2026_event_forecast_8e f
        CROSS JOIN dim_scenario_10a s
        LEFT JOIN dim_cocktail c
            ON c.cocktail_id = f.cocktail_id
        LEFT JOIN fact_cocktail_price p
            ON p.cocktail_id = f.cocktail_id
        LEFT JOIN v_cost_per_cocktail cp
            ON cp.cocktail_id = f.cocktail_id;
    """)

print("✅ 10B — v_2026_scenario_forecast_10b created")

✅ 10B — v_2026_scenario_forecast_10b created


#### 10C) Scenario Executive Monthly Summary

In [26]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_2026_scenario_exec_monthly_10c;")
    cur.execute("""
        CREATE VIEW v_2026_scenario_exec_monthly_10c AS
        SELECT
            scenario_name,
            year,
            month,
            SUM(scenario_qty) AS total_qty,
            ROUND(SUM(scenario_revenue), 2) AS total_revenue,
            ROUND(SUM(scenario_cogs), 2) AS total_cogs,
            ROUND(SUM(scenario_gross_profit), 2) AS total_gp
        FROM v_2026_scenario_forecast_10b
        GROUP BY scenario_name, year, month
        ORDER BY scenario_name, year, month;
    """)

print("✅ 10C — monthly scenario executive summary created")

✅ 10C — monthly scenario executive summary created


#### 10D) Scenario Executive Annual Summary

In [27]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_2026_scenario_exec_annual_10d;")
    cur.execute("""
        CREATE VIEW v_2026_scenario_exec_annual_10d AS
        SELECT
            scenario_name,
            SUM(scenario_qty) AS total_qty,
            ROUND(SUM(scenario_revenue), 2) AS total_revenue,
            ROUND(SUM(scenario_cogs), 2) AS total_cogs,
            ROUND(SUM(scenario_gross_profit), 2) AS total_gp
        FROM v_2026_scenario_forecast_10b
        GROUP BY scenario_name
        ORDER BY total_gp DESC;
    """)

print("✅ 10D — annual scenario executive summary created")

✅ 10D — annual scenario executive summary created


#### 10E) Delta vs BASE (Annual)

In [28]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_2026_scenario_delta_vs_base_10e;")
    cur.execute("""
        CREATE VIEW v_2026_scenario_delta_vs_base_10e AS
        WITH base AS (
            SELECT
                SUM(scenario_qty) AS base_qty,
                SUM(scenario_revenue) AS base_revenue,
                SUM(scenario_cogs) AS base_cogs,
                SUM(scenario_gross_profit) AS base_gp
            FROM v_2026_scenario_forecast_10b
            WHERE scenario_name = 'BASE'
        ),
        scen AS (
            SELECT
                scenario_name,
                SUM(scenario_qty) AS qty,
                SUM(scenario_revenue) AS revenue,
                SUM(scenario_cogs) AS cogs,
                SUM(scenario_gross_profit) AS gp
            FROM v_2026_scenario_forecast_10b
            GROUP BY scenario_name
        )
        SELECT
            s.scenario_name,
            s.qty,
            ROUND(s.revenue, 2) AS revenue,
            ROUND(s.cogs, 2) AS cogs,
            ROUND(s.gp, 2) AS gp,

            ROUND(s.qty - b.base_qty, 0) AS delta_qty,
            ROUND(s.revenue - b.base_revenue, 2) AS delta_revenue,
            ROUND(s.cogs - b.base_cogs, 2) AS delta_cogs,
            ROUND(s.gp - b.base_gp, 2) AS delta_gp
        FROM scen s
        CROSS JOIN base b
        ORDER BY delta_gp DESC;
    """)

print("✅ 10E — delta vs BASE (annual) created")

✅ 10E — delta vs BASE (annual) created


### 11) Sensitivity Analysis

#### 11A) Demand Sensitivity (+/- Footfall Shock)

In [29]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS dim_demand_sensitivity_11a (
            sensitivity_name TEXT PRIMARY KEY,
            demand_multiplier REAL NOT NULL,
            notes TEXT
        );
    """)

    cur.execute("DELETE FROM dim_demand_sensitivity_11a;")

    cur.execute("""
        INSERT INTO dim_demand_sensitivity_11a (sensitivity_name, demand_multiplier, notes)
        VALUES
            ('DOWN_10', 0.90, '-10% footfall'),
            ('DOWN_5',  0.95, '-5% footfall'),
            ('BASE',    1.00, 'No change'),
            ('UP_5',    1.05, '+5% footfall'),
            ('UP_10',   1.10, '+10% footfall');
    """)

    con.commit()

print("✅ 11A — demand sensitivity table created")

✅ 11A — demand sensitivity table created


#### 11B) Apply Demand Sensitivity to BASE Scenario

In [30]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_2026_demand_sensitivity_11b;")
    cur.execute("""
        CREATE VIEW v_2026_demand_sensitivity_11b AS
        SELECT
            d.sensitivity_name,
            f.cocktail_id,
            f.cocktail_name,
            f.year,
            f.month,

            ROUND(f.scenario_qty * d.demand_multiplier) AS adjusted_qty,

            ROUND(
                (f.scenario_sell_price * f.scenario_qty * d.demand_multiplier),
            2) AS adjusted_revenue,

            ROUND(
                (f.scenario_cost_per_cocktail * f.scenario_qty * d.demand_multiplier),
            2) AS adjusted_cogs,

            ROUND(
                (f.scenario_qty * d.demand_multiplier) *
                (f.scenario_sell_price - f.scenario_cost_per_cocktail),
            2) AS adjusted_gp

        FROM v_2026_scenario_forecast_10b f
        JOIN dim_demand_sensitivity_11a d
            ON f.scenario_name = 'BASE';
    """)

print("✅ 11B — demand sensitivity view created")

✅ 11B — demand sensitivity view created


#### 11C) Annual Impact Summary

In [31]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_2026_demand_sensitivity_exec_11c;")
    cur.execute("""
        CREATE VIEW v_2026_demand_sensitivity_exec_11c AS
        SELECT
            sensitivity_name,
            SUM(adjusted_qty) AS total_qty,
            ROUND(SUM(adjusted_revenue),2) AS total_revenue,
            ROUND(SUM(adjusted_cogs),2) AS total_cogs,
            ROUND(SUM(adjusted_gp),2) AS total_gp
        FROM v_2026_demand_sensitivity_11b
        GROUP BY sensitivity_name
        ORDER BY total_gp DESC;
    """)

print("✅ 11C — demand sensitivity annual summary created")

✅ 11C — demand sensitivity annual summary created


#### 11D) Dry January Stress Test

In [32]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_dry_january_stress_11d;")
    cur.execute("""
        CREATE VIEW v_dry_january_stress_11d AS
        SELECT
            f.cocktail_id,
            f.year,
            f.month,
            CASE 
                WHEN f.month = 1 THEN 
                    ROUND(f.event_adjusted_2026_qty * 0.60)   -- 40% drop
                ELSE 
                    f.event_adjusted_2026_qty
            END AS stressed_qty
        FROM fact_2026_event_forecast_8e f;
    """)

print("✅ 11D — Dry January stress test created")

✅ 11D — Dry January stress test created


#### 11E — Break-Even Sensitivity View

In [33]:
import sqlite3

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP VIEW IF EXISTS v_margin_per_unit_11e;")
    cur.execute("""
        CREATE VIEW v_margin_per_unit_11e AS
        SELECT
            c.cocktail_name,
            p.sell_price,
            cp.cost_per_cocktail,
            ROUND(p.sell_price - cp.cost_per_cocktail, 2) AS margin_per_unit,
            ROUND(
                (p.sell_price - cp.cost_per_cocktail) / p.sell_price,
            4) AS margin_ratio
        FROM dim_cocktail c
        JOIN fact_cocktail_price p
            ON p.cocktail_id = c.cocktail_id
        JOIN v_cost_per_cocktail cp
            ON cp.cocktail_id = c.cocktail_id
        ORDER BY margin_ratio ASC;
    """)

print("✅ 11E — margin per unit sensitivity view created")

✅ 11E — margin per unit sensitivity view created
